In [2]:
# Core FL and ML libraries
%pip install flwr[simulation] torch torchvision transformers datasets peft
# For efficiency during baseline training
%pip install bitsandbytes accelerate


  Using cached torchvision-0.26.0-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached flwr-1.27.0-py3-none-any.whl.metadata (14 kB)
  Using cached sqlalchemy-2.0.48-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.6-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached grpcio_health_checking-1.78.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached iterators-0.0.2-py3-none-any.whl.metadata (2.5 kB)
  Using cached pycryptodome-3.23.0-cp37-abi3-win_amd64.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached ray-2.51.1-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached tomli-2.4.1-cp312-cp312-win_amd64.whl.metadata (10 kB)
  Using cached tomli_w-1.2.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached msgpack-1.1.2-cp312-cp312-win_amd64.whl.metadata (8.4 kB)
  Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl.metadata (2.6 kB)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-core 3.1.3 requires packaging>=25.0, but you have packaging 24.2 which is incompatible.
googleapis-common-protos 1.63.2 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<4.0.0dev,>=3.19.5, but you have protobuf 6.33.6 which is incompatible.
proto-plus 1.24.0 requires protobuf<6.0.0dev,>=3.19.0, but you have protobuf 6.33.6 which is incompatible.
pyopenssl 24.3.0 requires cryptography<45,>=41.0.5, but you have c


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
#base model 
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
# Note: For the baseline, we use standard LoRA (PEFT) to make training feasible
config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")

In [2]:
import torch
import flwr as fl
from collections import OrderedDict

class SLMClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, val_loader, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device

    def get_parameters(self, config):
        """Returns only the trainable LoRA parameters."""
        return [val.cpu().numpy() for name, val in self.model.named_parameters() if val.requires_grad]

    def set_parameters(self, parameters):
        """Loads parameters into the trainable LoRA layers."""
        params_dict = zip([name for name, val in self.model.named_parameters() if val.requires_grad], parameters)
        state_dict = OrderedDict({k: torch.tensor(v).to(self.device) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=False)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        
        # Use AdamW for better LLM stability
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=2e-4)

        for batch in self.train_loader:
            # Move entire batch to GPU/CPU
            batch = {k: v.to(self.device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = self.model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

        return self.get_parameters(config={}), len(self.train_loader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        total_loss = 0.0
        
        with torch.no_grad():
            for batch in self.val_loader:
                batch = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**batch)
                total_loss += outputs.loss.item()
        
        # In LLM research, we often use Perplexity (exp of loss) instead of Accuracy
        return float(total_loss / len(self.val_loader)), len(self.val_loader.dataset), {}

In [6]:
from transformers import DataCollatorForLanguageModeling

def prepare_dataloaders(partition, tokenizer, batch_size=2):
    """
    Generalized data preparer that takes any Hugging Face partition 
    and returns PyTorch Loaders.
    """
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

    tokenized_dataset = partition.map(tokenize_function, batched=True, remove_columns=partition.column_names)
    
    # Data collator handles dynamic padding for LLMs
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    train_loader = torch.utils.data.DataLoader(
        tokenized_dataset, batch_size=batch_size, collate_fn=data_collator
    )
    return train_loader

In [7]:
import ray
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# This prevents 'Actor' errors and ensures the simulation can restart if it crashes
ray.init(ignore_reinit_error=True, num_cpus=2)

# 1. SETUP (Always First)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. MODEL LOADING
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
base_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"": 0})
config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
model = get_peft_model(base_model, config)

# 3. DATA & CLIENT LOGIC (Referencing the model above)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
dataset = dataset.filter(lambda x: len(x["text"]) > 10)   #filter empty lines for better training
partition_size = len(dataset) // 5  # Assuming 5 clients

def client_fn(cid: str):
    """
    Create a Flower client representing a single device.
    'cid' is the unique ID for the client (0, 1, 2...).
    """
    # Create the local partition for this specific client
    idx = int(cid)
    start_idx = idx * partition_size
    end_idx = (idx + 1) * partition_size
    client_partition = dataset.select(range(start_idx, end_idx))
    
    # Split into train/val (90/10)
    client_partition = client_partition.train_test_split(test_size=0.1)
    
    # Prepare DataLoaders using the generalized function we discussed
    train_loader = prepare_dataloaders(client_partition["train"], tokenizer)
    val_loader = prepare_dataloaders(client_partition["test"], tokenizer)
    
    # Return the SLMClient instance
    # Note: 'model' and 'device' should be defined in your main script scope
    return SLMClient(model, train_loader, val_loader, device)




2026-03-28 19:12:51,849	INFO worker.py:2012 -- Started a local Ray instance.
f:\Python\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

f:\Python\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in F:\Huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
(pid=gcs_server) [2026-03-28 19:13:03,682 E 9432 20660] (gcs_server.exe) gcs_server.cc:302: Failed to establish connection to the event+metrics

RuntimeError: CUDA is required but not available for bitsandbytes. Please consider installing the multi-platform enabled version of bitsandbytes, which is currently a work in progress. Please check currently supported platforms and installation instructions at https://huggingface.co/docs/bitsandbytes/main/en/installation#multi-backend

(raylet) [2026-03-28 19:13:22,882 E 8892 14272] (raylet.exe) main.cc:975: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


In [ ]:
# Plotting Functions
import matplotlib.pyplot as plt

def plot_federated_learning_results(history):
    """
    Visualizes the training loss over federated rounds.
    """
    # Extracting loss from history
    rounds, losses = zip(*history.losses_distributed)
    
    plt.figure(figsize=(10, 6))
    plt.plot(rounds, losses, marker='o', linestyle='-', color='b', label='Baseline (FedAvg)')
    plt.title('Federated Learning Training Loss (TinyLlama Baseline)')
    plt.xlabel('Federated Round')
    plt.ylabel('Distributed Loss')
    plt.grid(True)
    plt.legend()
    
    # Save the plot for your thesis/report
    plt.savefig("federated_learning_loss.png")
    plt.show()

In [ ]:
# 4. SIMULATION START
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=5,
    config=fl.server.ServerConfig(num_rounds=3, round_timeout=1800.0),
    strategy=fl.server.strategy.FedAvg(),
    client_resources={"num_gpus": 0.4, "num_cpus": 2} # Use 0.4 to ensure 2 clients don't try to run at once
)

plot_federated_learning_results(history)

In [4]:
import torch
import flwr as fl

# Flower client for local training/eval
class HealthClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, val_loader, device="cpu"):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device

    def get_parameters(self):
        return [val.cpu().numpy() for val in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        state_dict = {
            k: torch.tensor(v).to(self.device)
            for k, v in zip(self.model.state_dict().keys(), parameters)
        }
        self.model.load_state_dict(state_dict, strict=False)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.Adam(self.model.parameters(), lr=float(config.get("lr", 1e-4)))
        loss_fn = torch.nn.CrossEntropyLoss()

        epochs = int(config.get("epochs", 1))
        for _ in range(epochs):
            for batch in self.train_loader:
                # adapt tensor unpacking to your dataloader output
                x, y = batch
                x, y = x.to(self.device), y.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(x)
                loss = loss_fn(outputs.logits if hasattr(outputs, "logits") else outputs, y)
                loss.backward()
                optimizer.step()

        return self.get_parameters(), len(self.train_loader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()

        loss_fn = torch.nn.CrossEntropyLoss()
        loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in self.val_loader:
                x, y = batch
                x, y = x.to(self.device), y.to(self.device)
                outputs = self.model(x)
                logits = outputs.logits if hasattr(outputs, "logits") else outputs
                loss += loss_fn(logits, y).item() * x.size(0)
                preds = logits.argmax(dim=-1)
                correct += (preds == y).sum().item()
                total += x.size(0)

        return float(loss / total), total, {"accuracy": float(correct / total)}

# Server strategy: FedAvg
strategy = fl.server.strategy.FedAvg()

# (Optional) Start Flower server
# fl.server.start_server(
#     server_address="0.0.0.0:8080",
#     config=fl.server.ServerConfig(num_rounds=3),
#     strategy=strategy,
# )